### lesson 01: embbeding a text (vectorization)

In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
## this model is normalized, so we can use dot product to compare the similarity between two vectors
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
## embbeding a question
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
## embedding an answer
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
## compare it using dot product
v1.dot(dv)

np.float32(0.32332402)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
## comparing unrelated one
v2.dot(dv)

np.float32(0.019730464)

### lesson 02: vectorizing the documents

In [ ]:
# at top of your script/notebook in week_2
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))


In [9]:
from src.retrieve_documents import retrieve_documents

In [10]:
documents = retrieve_documents()

Number of documents retrieved: 1350


In [11]:
## create the list to make emmbedings
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
from tqdm.auto import tqdm

In [13]:
## creating the vectors!
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
## convert to numpy to folow this examples
import numpy as np
X = np.array(vectors)

### lesson 3: vectorsearch

In [15]:
## first encode a query to search it on the vector space
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [16]:
## vector search using numpy, computing as a matrix
scores = X.dot(v_query)
scores

array([ 0.487406  ,  0.20991938,  0.762941  , ..., -0.08637968,
        0.03759793, -0.03037042], shape=(1350,), dtype=float32)

In [17]:
## get the scores using a list of comprehension
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [18]:
## showing the index and the dot product of the best match
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294094))

In [19]:
## review the document
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [20]:
## retrieve the top 5
top5 = np.argsort(scores)[-5:]

In [21]:
## as they come from the smallest first,reverse it to get the largest first
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [22]:
type(scores)

list

In [23]:
np.array(scores)[top5]

array([0.76294094, 0.7579372 , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [24]:
## get the top 5 index in a shorter way
top5 = np.argsort(-np.array(scores))[:5]
top5

array([  2, 625, 907, 538,   7])

In [25]:
## review it
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294094
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

### Minsearch vector search

In [26]:
from minsearch import VectorSearch

In [27]:
## load the numpy vectors and the documents
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [28]:
## vectir search
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [29]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [30]:
## reducing the filtering with keywords
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [31]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

### RAG using the vector space from minsearch

In [33]:
from langchain_ollama import ChatOllama

from src.rag_helper import RAGBase


In [34]:
## creating a class that inherits the past RAG class
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [ ]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()


vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=ChatOllama,
    model="qwen3.5:0.8b",
    instructions=instructions
)

In [36]:
vector_assistant.rag("the program has already begun, can I still sign up?")

"Based on the provided context, I can answer your question. The most relevant information in the context is from the section about the Zoomcamp course:\n\n**Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?**\n\n**A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.**\n\nThis answer indicates that the course has begun and you are already accepted, so your question about whether you can still sign up is:\n\n**Yes**\n\nHowever, based on the context, it's important to note that while registration can still be done after the program starts, you're currently not required to announce or reserve your project idea for the Capstone Project section."

### using persistent vector search

In [37]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

sqlitesearch supports three ANN modes:

- `lsh` (default): up to 100K vectors, random hyperplane projections
- `ivf`: 10K-500K vectors, K-means clustering
- `hnsw`: 10K-1M+ vectors, proximity graph (highest recall)

In [38]:
vs_index.fit(vectors, documents)

In [40]:
## searching for the documents

query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [41]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [42]:
## filter by course (or keyword)
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [43]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [44]:
## closing the connection
vs_index.close()


In [46]:
## combination of sentence transformers and vector search
model = SentenceTransformer("all-MiniLM-L6-v2")

## opening the pre computed emmbedings
vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [47]:
query_vector = model.encode("How do I run Kafka?")
results = vs_index.search(query_vector, num_results=5)

In [50]:
## re using the RAGVector class
instructions = """Role: an amazing assistant
task: answer the questions polite using the context provided
steps: 1. analyze the question. 2. analyze the context.
3. If the context does not match with the just say 'I don't now because these is not information about it'.
4. Give the answer if is it on the context, concise and polite
output: the answer of the question"""

vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=ChatOllama,
    model="batiia-gemma-mini:latest",
    instructions=instructions
)

In [51]:
vector_assistant.rag("the program has already begun, can I still sign up?")

"I don't know."

In [52]:
vs_index.close()